In [15]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

def get_robust_session():
    session = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=2, # Increased wait time between retries
        status_forcelist=[500, 502, 503, 504],
        raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('https://', adapter)
    return session

def collect_politics_links():
    OUTPUT_FILE = "goobjoog_siyaasad_links.xlsx"
    PAGES_TO_SCAN = 20  # Total pages to look through
    session = get_robust_session()
    all_links = set()

    # Load existing links if they exist to avoid duplicates
    if os.path.exists(OUTPUT_FILE):
        existing_df = pd.read_excel(OUTPUT_FILE)
        all_links = set(existing_df['URL'].tolist())
        print(f"🔄 Loaded {len(all_links)} existing links.")

    # Headers strictly matching your successful curl
    headers = {
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36',
        'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'referer': 'https://goobjoog.com/?s='
    }

    print("🚀 Starting link collection...")

    for page in range(1, PAGES_TO_SCAN + 1):
        # Apply the exact URL structure you identified
        page_url = "https://goobjoog.com/?s=" if page == 1 else f"https://goobjoog.com/page/{page}/?s="
        
        print(f"🌐 Scanning Page {page}: {page_url}")
        
        try:
            response = session.get(page_url, headers=headers, timeout=30)
            
            if response.status_code != 200:
                print(f"🛑 Error: Status {response.status_code}. Stopping.")
                break
                
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find all article links (rel='bookmark' is standard for these results)
            found_tags = soup.find_all('a', rel='bookmark')
            
            new_count = 0
            for tag in found_tags:
                url = tag.get('href')
                # Filter for actual articles and ensure it's unique
                if url and "/20" in url and url not in all_links:
                    all_links.add(url)
                    new_count += 1
            
            print(f"   ✅ Found {new_count} new links. Total: {len(all_links)}")
            
            # Save progress every page
            pd.DataFrame(list(all_links), columns=["URL"]).to_excel(OUTPUT_FILE, index=False)
            
            # Pause between pages to keep the connection alive
            time.sleep(5) 

        except Exception as e:
            print(f"❌ Connection dropped on page {page}: {e}")
            print("🕒 Waiting 30 seconds to let the server reset...")
            time.sleep(30)
            continue

    print(f"\n✨ DONE! Collected {len(all_links)} links in {OUTPUT_FILE}")

if __name__ == "__main__":
    collect_politics_links()

🚀 Starting link collection...
🌐 Scanning Page 1: https://goobjoog.com/?s=
❌ Connection dropped on page 1: HTTPSConnectionPool(host='goobjoog.com', port=443): Max retries exceeded with url: /?s= (Caused by ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)))
🕒 Waiting 30 seconds to let the server reset...
🌐 Scanning Page 2: https://goobjoog.com/page/2/?s=


KeyboardInterrupt: 

In [18]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

def get_robust_session():
    session = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[500, 502, 503, 504],
        raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

def scrape_goobjoog_article(url, session):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    }
    
    try:
        response = session.get(url, headers=headers, timeout=20)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # 1. Extract Headline
            headline_tag = soup.find('h1', class_='entry-title') or soup.find('h1')
            headline = headline_tag.get_text(strip=True) if headline_tag else "No Headline"
            
            # 2. Extract Body Content
            body_content = ""
            article_div = soup.find('div', class_='entry-content')
            
            if article_div:
                # Clean HTML
                for extra in article_div(['script', 'style', 'aside', 'ins']):
                    extra.decompose()
                
                paragraphs = article_div.find_all('p')
                body_content = "\n".join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
            
            return headline, body_content
        else:
            return None, f"Status {response.status_code}"
    except Exception as e:
        return None, str(e)

# --- EXECUTION ---

# 1. Update filenames for your Siyaasad project
input_file = "goobjoog_siyaasad_links.xlsx"
output_file = "goobjoog_siyaasad_scraped_articles.xlsx"

if not os.path.exists(input_file):
    print(f"❌ Error: {input_file} not found.")
    exit()

df_links = pd.read_excel(input_file)
urls_to_scrape = df_links['URL'].tolist() 

# 2. Setup Resume/Checkpoint Logic
scraped_data = []
processed_urls = set()

if os.path.exists(output_file):
    try:
        df_existing = pd.read_excel(output_file)
        scraped_data = df_existing.to_dict('records')
        # Check if column is 'url' or 'URL'
        col = 'url' if 'url' in df_existing.columns else 'URL'
        processed_urls = set(df_existing[col].tolist())
        print(f"🔄 Resuming: {len(processed_urls)} articles already scraped.")
    except Exception as e:
        print(f"Starting fresh dataset. (Error loading existing: {e})")

# 3. Main Loop
session = get_robust_session()
print(f"🚀 Starting scrape of {len(urls_to_scrape)} articles...")

for i, url in enumerate(urls_to_scrape):
    if url in processed_urls:
        continue
    
    print(f"[{i+1}/{len(urls_to_scrape)}] Scrapping: {url}")
    
    headline, body = scrape_goobjoog_article(url, session)
    
    scraped_data.append({
        'url': url,
        'headline': headline,
        'body': body,
        'source': 'goobjoog',
        'category': 'siyaasad'
    })
    processed_urls.add(url)

    # Save every 5 articles to prevent data loss
    if len(scraped_data) % 5 == 0 and len(scraped_data) > 0:
        pd.DataFrame(scraped_data).to_excel(output_file, index=False)
    
    # Stay slow to avoid 10054 errors
    time.sleep(3.0)

# Final Save
pd.DataFrame(scraped_data).to_excel(output_file, index=False)
print(f"\n✅ Scraping Complete! {len(scraped_data)} articles saved to {output_file}")

Starting fresh dataset. (Error loading existing: 'URL')
🚀 Starting scrape of 1250 articles...
[1/1250] Scrapping: https://goobjoog.com/2026/02/11/muxuu-ka-dhahay-madaxweyne-xasan-sheekh-maxamuud-guusha-soomaaliya-ee-kursiga-golaha-nabadda-iyo-amniga-midowga-afrika/
[2/1250] Scrapping: https://goobjoog.com/2026/02/13/wafdi-uu-hoggaaminayo-madaxweyne-xasan-sheekh-oo-addis-ababa-gaaray/
[3/1250] Scrapping: https://goobjoog.com/2026/04/29/qiimaha-shidaalka-caalamka-oo-si-kordhaya-iyo-maraykanka-oo-la-sheegay-inuu-sabab-u-yahay/
[4/1250] Scrapping: https://goobjoog.com/2025/12/30/difaaca-qarannimada-soomaaliya-iyo-dardargelinta-shidaalka-nuxurka-shirka-jaraaid-ee-madaxweynayaasha-soomaaliya-iyo-turkiga/
[5/1250] Scrapping: https://goobjoog.com/2025/12/03/dfs-oo-ka-hadashay-hadalladii-madaxweyne-donald-trump-uu-ku-aflagaadeeyay-soomaaliya/
[6/1250] Scrapping: https://goobjoog.com/2025/12/22/trump-oo-shaqada-ka-saaray-30-danjire-oo-ay-ku-jiraan-safiirkii-mareykanka-ee-soomaaliya/
[7/1250] Scr

PermissionError: [Errno 13] Permission denied: 'goobjoog_siyaasad_scraped_articles.xlsx'

In [19]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

def get_robust_session():
    session = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[500, 502, 503, 504],
        raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

def scrape_somali_news(url, session):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    try:
        response = session.get(url, headers=headers, timeout=20)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # 1. Get the Headline
            headline = soup.find('h1').get_text(strip=True) if soup.find('h1') else "No Headline Found"
            
            # 2. Get the Body using your updated logic
            body_content = ""
            article_div = soup.find('div', class_='entry-content') or \
                          soup.find('div', class_='article-content') or \
                          soup.find('div', id='content-main')
            
            if article_div:
                paragraphs = article_div.find_all('p')
                body_content = "\n".join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
            else:
                paragraphs = soup.find_all('p')
                body_content = "\n".join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])

            return headline, body_content
        else:
            return None, f"Failed Status: {response.status_code}"
    except Exception as e:
        return None, str(e)

# --- CONFIGURATION ---
input_file = "goobjoog_siyaasad_links.xlsx"
output_file = "goobjoog_siyaasad_scraped_articles.xlsx"

if not os.path.exists(input_file):
    print(f"❌ Error: {input_file} not found.")
else:
    df_links = pd.read_excel(input_file)
    urls_to_scrape = df_links['URL'].tolist() 

    # --- RESUME LOGIC (CHECKPOINT) ---
    scraped_data = []
    processed_urls = set()

    if os.path.exists(output_file):
        try:
            df_existing = pd.read_excel(output_file)
            scraped_data = df_existing.to_dict('records')
            # Handle both 'url' and 'URL' column naming
            col = 'url' if 'url' in df_existing.columns else 'URL'
            processed_urls = set(df_existing[col].tolist())
            print(f"🔄 Resuming: {len(processed_urls)} articles already in {output_file}")
        except Exception as e:
            print(f"Starting fresh. (Error loading existing file: {e})")

    # --- MAIN EXECUTION ---
    session = get_robust_session()
    print(f"🚀 Starting scrape of {len(urls_to_scrape)} URLs...")

    # For testing, you can slice this: urls_to_scrape[:5]
    for i, url in enumerate(urls_to_scrape):
        if url in processed_urls:
            continue
            
        print(f"[{i+1}/{len(urls_to_scrape)}] Processing: {url}")
        
        headline, body = scrape_somali_news(url, session)
        
        # Validation for your NLP dataset
        if headline and body and len(body) > 100:
            scraped_data.append({
                'url': url,
                'headline': headline,
                'body': body,
                'source': 'goobjoog',
                'category': 'siyaasad'
            })
            processed_urls.add(url)
            print(f"  ✅ Saved: {headline[:40]}...")
        else:
            print(f"  ⚠️ Skipped: Missing or insufficient content.")

        # CHECKPOINT: Save every 5 new articles
        if len(scraped_data) % 5 == 0 and len(scraped_data) > 0:
            pd.DataFrame(scraped_data).to_excel(output_file, index=False)
            print(f"💾 Checkpoint saved: {len(scraped_data)} total articles.")

        time.sleep(2.0) # To avoid 10054 Connection reset errors

    # FINAL SAVE
    if scraped_data:
        pd.DataFrame(scraped_data).to_excel(output_file, index=False)
        print(f"\n✅ Done! Total {len(scraped_data)} articles saved to {output_file}")
    else:
        print("\n❌ No data was captured.")

🚀 Starting scrape of 1250 URLs...
[1/1250] Processing: https://goobjoog.com/2026/02/11/muxuu-ka-dhahay-madaxweyne-xasan-sheekh-maxamuud-guusha-soomaaliya-ee-kursiga-golaha-nabadda-iyo-amniga-midowga-afrika/
  ✅ Saved: Muxuu ka dhahay Madaxweyne Xasan Sheekh ...
[2/1250] Processing: https://goobjoog.com/2026/02/13/wafdi-uu-hoggaaminayo-madaxweyne-xasan-sheekh-oo-addis-ababa-gaaray/
  ✅ Saved: Wafdi uu Hoggaaminayo Madaxweyne Xasan S...
[3/1250] Processing: https://goobjoog.com/2026/04/29/qiimaha-shidaalka-caalamka-oo-si-kordhaya-iyo-maraykanka-oo-la-sheegay-inuu-sabab-u-yahay/
  ✅ Saved: Qiimaha Shidaalka Caalamka oo si kordhay...
[4/1250] Processing: https://goobjoog.com/2025/12/30/difaaca-qarannimada-soomaaliya-iyo-dardargelinta-shidaalka-nuxurka-shirka-jaraaid-ee-madaxweynayaasha-soomaaliya-iyo-turkiga/
  ✅ Saved: Difaaca Qarannimada Soomaaliya iyo Darda...
[5/1250] Processing: https://goobjoog.com/2025/12/03/dfs-oo-ka-hadashay-hadalladii-madaxweyne-donald-trump-uu-ku-aflagaadeeyay-s